# RL PCB Benchmark Comparison
This notebook pulls the EDA_PCB_RL_Benchmark repository, sets up the environment on Google Colab, trains three RL models (PPO, TD3, SAC) for 10,000 timesteps each, and compiles the results to draw a conclusion.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/Alli-Ekundayo/EDA_PCB_RL_Benchmark.git
%cd EDA_PCB_RL_Benchmark

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt
!pip install torch-geometric reportlab pandas matplotlib
# Disable virtual environment sourcing in the bash scripts as Colab handles packages globally
!sed -i 's/source .venv\/bin\/activate/# source .venv\/bin\/activate/g' tests/*/run.sh

In [ ]:
# 3. Run PPO Benchmark
!bash tests/01_ppo_research_run/run.sh

In [ ]:
# 4. Run TD3 Benchmark
!bash tests/02_td3_research_run/run.sh

In [ ]:
# 5. Run SAC Benchmark
!bash tests/03_sac_research_run/run.sh

In [ ]:
# 6. Compile Results and Draw Conclusion
import re
import pandas as pd
from IPython.display import display, Markdown

def parse_log(log_path):
    try:
        rewards = []
        with open(log_path, 'r') as f:
            for line in f:
                match = re.search(r'train/mean_reward=([-+]?\d*\.\d+|\d+)', line)
                if match:
                    rewards.append(float(match.group(1)))
        return rewards[-1] if rewards else None
    except Exception:
        return None

results = {
    "PPO": parse_log("tests/01_ppo_research_run/results/training.log"),
    "TD3": parse_log("tests/02_td3_research_run/results/training.log"),
    "SAC": parse_log("tests/03_sac_research_run/results/training.log"),
}

df = pd.DataFrame(list(results.items()), columns=["Algorithm", "Final Mean Reward"])
display(Markdown("### Final Benchmark Comparison"))
display(df)

best_algo = df.loc[df['Final Mean Reward'].idxmax(), 'Algorithm']
conclusion = f"""
### Conclusion
Based on the 10,000 timestep training run, the **{best_algo}** algorithm achieved the highest final mean reward. 
This indicates its relative stability and performance for the EDA PCB routing task over this horizon.
"""
display(Markdown(conclusion))